In [2]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle

# Load model and label encoder
model = load_model("D:\\Egyptian money detection\\combined_sift_hog_model.h5")  # or "model_sift.h5", "model_hog.h5"

with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# Preprocessing (same as used in training)
def apply_combined_sift_hog(image):
    def apply_sift(img, max_keypoints=100):
        sift = cv2.SIFT_create(nfeatures=max_keypoints)
        keypoints, descriptors = sift.detectAndCompute(img, None)
        sift_image = np.zeros_like(img)
        sift_image = cv2.drawKeypoints(img, keypoints, sift_image, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
        return cv2.cvtColor(sift_image, cv2.COLOR_BGR2RGB)

    def apply_hog(img):
        from skimage.feature import hog
        from skimage import color
        gray = color.rgb2gray(img)
        _, hog_image = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                           cells_per_block=(2, 2), block_norm='L2-Hys',
                           visualize=True)
        hog_image = (hog_image * 255).astype(np.uint8)
        return cv2.cvtColor(hog_image, cv2.COLOR_GRAY2RGB)

    sift_img = apply_sift(image)
    hog_img = apply_hog(image)
    return cv2.addWeighted(sift_img, 0.5, hog_img, 0.5, 0)

# Real-time camera
cap = cv2.VideoCapture(0)

print("📷 Starting real-time prediction. Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img_resized = cv2.resize(frame, (224, 224))
    processed = apply_combined_sift_hog(img_resized)
    input_img = processed.astype('float32') / 255.0
    input_img = np.expand_dims(input_img, axis=0)

    pred = model.predict(input_img)
    label = le.classes_[np.argmax(pred)]

    cv2.putText(frame, f"Prediction: {label}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow("Real-Time Prediction", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


FileNotFoundError: [Errno 2] No such file or directory: 'label_encoder.pkl'